# Fine-Tune an LLM with the SageMaker SDK and TRL

This notebook shows how to fine-tune a small language model on Amazon SageMaker AI with the SageMaker Python SDK `ModelTrainer` and a [TRL](https://huggingface.co/docs/trl) `SFTTrainer` training script. The job runs on the Hugging Face PyTorch Training DLC, which ships with Transformers, TRL, Datasets, and PyTorch pre-installed.

**Hardware:** `ml.g6.xlarge` (1x NVIDIA L4) — one GPU is enough for a 0.6B model.

You will learn how to:

- Write a training script that reads hyperparameters and SageMaker environment variables.
- Retrieve the Hugging Face training DLC for your region and instance type.
- Create a `ModelTrainer` with CloudWatch metrics and a runtime safety cap.
- Start the training job and locate the trained model artifacts in S3.

## Setup

Install the SageMaker Python SDK v3:

In [1]:
%pip install "sagemaker>=3.0.0" --upgrade --quiet

/Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


> [!NOTE]
> This example uses the [SageMaker Python SDK v3](https://github.com/aws/sagemaker-python-sdk). v3 introduces a new, framework-agnostic API built around `ModelBuilder` (inference) and `ModelTrainer` (training), which replaces the v2 `HuggingFaceModel` and `HuggingFace` classes.

## Session and execution role

The training job runs under an IAM execution role with access to S3. Inside SageMaker Studio or a notebook instance, `get_execution_role()` finds the role automatically. Locally, the role is looked up by name — adjust `role_name` to your setup.

In [2]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

REGION = boto3.Session().region_name or "us-east-1"
boto_sess = boto3.Session(region_name=REGION)
sess = Session(boto_session=boto_sess)

try:
    role = get_execution_role(sagemaker_session=sess)
    print(f"Using the SageMaker execution role: {role}")
except Exception:
    role_name = "sagemaker_execution_role"
    role = boto_sess.client("iam").get_role(RoleName=role_name)["Role"]["Arn"]
    print(f"Using the IAM role: {role}")

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/dwarez/Library/Application Support/sagemaker/config.yaml


[08/17/26 16:45:59] INFO     Loading cached SSO token for hf                                          ]8;id=3013213;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=3013214;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/botocore/tokens.py#377\377]8;;\

Using the SageMaker execution role: arn:aws:iam::754289655784:role/aws-reserved/sso.amazonaws.com/AWSReservedSSO_HF-Sandbox-access_a9a3037b77bf6782


In [3]:
role = "arn:aws:iam::754289655784:role/sagemaker_execution_role"

## The training script

The `ModelTrainer` runs your own script inside the training container. The script for this notebook lives in [`scripts/train.py`](https://github.com/huggingface/hub-docs/tree/main/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/scripts/train.py) and does three things:

1. Reads the hyperparameters as command-line arguments (`--model_name`, `--max_steps`, ...).
2. Loads the dataset from the Hugging Face Hub and fine-tunes the model with TRL `SFTTrainer`.
3. Saves the model and tokenizer to `SM_MODEL_DIR`, which SageMaker archives to S3 as `model.tar.gz` when the job finishes.

## Create the ModelTrainer

The `ModelTrainer` ties everything together:

- `source_code` points at the script directory and entry point.
- `compute` defines the instance. `enable_managed_spot_training=True` uses [managed spot instances](https://docs.aws.amazon.com/sagemaker/latest/dg/model-managed-spot-training.html) for up to 90% savings — fine here because the job takes a few minutes.
- `training_image` is the Hugging Face training DLC, retrieved for your region and instance type with `image_uris.retrieve`.
- `with_metric_definitions` parses the training logs and sends metrics to CloudWatch.

In [8]:
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, StoppingCondition, MetricDefinition
from sagemaker.core import image_uris

hyperparameters = {
    "model_name": "Qwen/Qwen3-0.6B",       # any small causal LM from the Hub works
    "dataset_name": "trl-lib/Capybara",    # conversational SFT dataset
    "max_steps": 50,                       # short run: enough to see the loss go down
    "train_batch_size": 4,
    "learning_rate": 2e-5,
}

instance_type = "ml.p4de.24xlarge"

# Retrieve the Hugging Face PyTorch training DLC image URI
training_image = image_uris.retrieve(
    framework="huggingface",
    region=REGION,
    version="5.3.0",                         # Transformers version
    base_framework_version="pytorch2.9.0",   # PyTorch version
    py_version="py312",                      # Python version
    image_scope="training",
    instance_type=instance_type,
)

# SFTTrainer logs lines like {'loss': 2.34, ...}; parse the loss into CloudWatch
metric_definitions = [
    MetricDefinition(name="train-loss", regex="'loss': ([0-9.]+)"),
]

model_trainer = ModelTrainer(
    sagemaker_session=sess,
    role=role,
    training_image=training_image,
    source_code=SourceCode(
        source_dir="./scripts",            # directory with the training script
        entry_script="train.py",           # script to run in the training job
    ),
    compute=Compute(
        instance_type=instance_type,
        instance_count=1,
        enable_managed_spot_training=True,  # use managed spot instances
    ),
    # max_wait_time_in_seconds should be equal to or greater than max_runtime_in_seconds
    stopping_condition=StoppingCondition(
        max_runtime_in_seconds=3600,
        max_wait_time_in_seconds=7200,
    ),
    hyperparameters=hyperparameters,
).with_metric_definitions(metric_definitions)

[08/17/26 16:53:51] INFO     Base name not provided. Using default name:                            ]8;id=3013303;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=3013304;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py#129\129]8;;\
                             huggingface-pytorch-training-job                                                      

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=3013309;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=3013310;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py#192\192]8;;\
                             s3_output_path='s3://sagemaker-us-east-1-754289655784/huggingface-pyto                
                             rch-training-job' kms_key_id=None compression_type='GZIP'                             

                    INFO     Training image URI:                                               ]8;id=3013315;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=3013316;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-                     
                             training:2.9.0-transformers5.3.0-gpu-py312-cu130-ubuntu22.04                          

## Start the training job

Call `train` to launch the job. SageMaker starts the instance, runs `train.py` with your hyperparameters, streams the logs, and uploads the model artifacts to S3 when done. The dataset downloads from the Hub inside the container, so there is no data to upload.

In [9]:
model_trainer.train()

[08/17/26 16:54:06] INFO     Training Job Name:                                                ]8;id=3013321;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=3013322;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#823\823]8;;\
                             huggingface-pytorch-training-job-20260817165353                                       

                    INFO     Creating training_job resource.                                     ]8;id=3013327;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3013328;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#31238\31238]8;;\

Output()

[08/17/26 16:54:08] INFO     Loading cached SSO token for hf                                          ]8;id=3013333;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=3013334;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/botocore/tokens.py#377\377]8;;\

[08/17/26 17:40:35] INFO     huggingface-pytorch-training-job-20260817165353/algo-1-1786981019:  ]8;id=3013340;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3013341;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             Starting training script                                                              

...

[08/17/26 17:42:32] INFO     huggingface-pytorch-training-job-20260817165353/algo-1-1786981019:  ]8;id=3016562;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3016563;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             30%|███       | 15/50 [00:42<01:36,  2.74s/it]#015 32%|███▏      |                    
                             16/50 [00:45<01:31,  2.69s/it]#015 34%|███▍      | 17/50                              
                             [00:48<01:30,  2.73s/it]#015 36%|███▌      | 18/50 [00:51<01:28,                      
                             2.78s/it]#015 38%|███▊      | 19/50 [00:53<01:24,  2.72s/it]#015                      
                             40%|████      | 20/50 [00:56<01:23,  2.79s/it]#015                                    
                             #015{'loss': '1.461', 'grad_norm': '2.484', 'learning_rate':                          
                             '1.24e-05', 'entropy': '1.458', 'num_tokens': '4.488e+05',                            
                             'mean_token_accuracy': '0.6485', 'epoch': '0.04049'}                                  

[08/17/26 17:42:44] INFO     huggingface-pytorch-training-job-20260817165353/algo-1-1786981019:  ]8;id=3016568;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3016569;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             40%|████      | 20/50 [00:56<01:23,  2.79s/it]#015 42%|████▏     |                    
                             21/50 [00:59<01:19,  2.76s/it]#015 44%|████▍     | 22/50                              
                             [01:02<01:18,  2.81s/it]#015 46%|████▌     | 23/50 [01:05<01:14,                      
                             2.76s/it]#015 48%|████▊     | 24/50 [01:07<01:12,  2.80s/it]#015                      
                             50%|█████     | 25/50 [01:10<01:11,  2.84s/it]#015                                    
                             #015{'loss': '1.421', 'grad_norm': '2.134', 'learning_rate':                          
                             '1.04e-05', 'entropy': '1.418', 'num_tokens': '5.598e+05',                            
                             'mean_token_accuracy': '0.6497', 'epoch': '0.05061'}                                  

[08/17/26 17:42:56] INFO     huggingface-pytorch-training-job-20260817165353/algo-1-1786981019:  ]8;id=3016574;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3016575;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             50%|█████     | 25/50 [01:10<01:11,  2.84s/it]#015 52%|█████▏    |                    
                             26/50 [01:13<01:06,  2.76s/it]#015 54%|█████▍    | 27/50                              
                             [01:16<01:03,  2.78s/it]#015 56%|█████▌    | 28/50 [01:18<00:59,                      
                             2.72s/it]#015 58%|█████▊    | 29/50 [01:21<00:57,  2.76s/it]#015                      
                             60%|██████    | 30/50 [01:24<00:54,  2.73s/it]#015                                    
                             #015{'loss': '1.39', 'grad_norm': '2.187', 'learning_rate':                           
                             '8.4e-06', 'entropy': '1.38', 'num_tokens': '6.744e+05',                              
                             'mean_token_accuracy': '0.66', 'epoch': '0.06073'}                                    

[08/17/26 17:43:13] INFO     huggingface-pytorch-training-job-20260817165353/algo-1-1786981019:  ]8;id=3016580;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3016581;file:///Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             60%|██████    | 30/50 [01:24<00:54,  2.73s/it]#015 62%|██████▏   |                    
                             31/50 [01:27<00:52,  2.76s/it]#015 64%|██████▍   | 32/50                              
                             [01:29<00:50,  2.78s/it]#015 66%|██████▌   | 33/50 [01:32<00:46,                      
                             2.72s/it]#015 68%|██████▊   | 34/50 [01:35<00:44,  2.75s/it]#015                      
                             70%|███████   | 35/50 [01:37<00:40,  2.71s/it]#015                                    
                             #015{'loss': '1.517', 'grad_norm': '2.092', 'learning_rate':                          
                             '6.4e-06', 'entropy': '1.51', 'num_tokens': '7.901e+05',                              
                             'mean_token_accuracy': '0.641', 'epoch': '0.07085'}                                   

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 model_trainer.train()                                                                        │
│   2                                                                                              │
│                                                                                                  │
│ /Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/l │
│ ib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py:403 in wrapper         │
│                                                                                                  │
│   400 │   │   │   │   │   caught_ex = e                                                          │
│   401 │   │   │   │   finally:                                                                   │
│   402 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 403 │   │   │   │   │   │   raise caught_ex                                                    │
│   404 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   405 │   │   │   else:                                                                          │
│   406 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/l │
│ ib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py:368 in wrapper         │
│                                                                                                  │
│   365 │   │   │   │   start_timer = perf_counter()                                               │
│   366 │   │   │   │   try:                                                                       │
│   367 │   │   │   │   │   # Call the original function                                           │
│ ❱ 368 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   369 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   370 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   371 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sagemaker-sdk/fine-tune-llm-sft/.venv/l │
│ ib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:346 in wrapper           │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /Users/dwarez/hf/repos/hub-docs/docs/sagemaker/notebooks/sa

## The trained model

When the job completes, the model artifacts (`model.tar.gz`) are in S3:

In [ ]:
model_data = model_trainer._latest_training_job.model_artifacts.s3_model_artifacts
print(f"Trained model artifacts: {model_data}")

## What's next

Deploy the trained model to an endpoint with `ModelBuilder` by pointing it at this S3 URI — see [Deploy models](https://huggingface.co/docs/sagemaker/main/en/tutorials/sagemaker-sdk/deploy-sagemaker-sdk) for the full guide.